In [1]:
# Jaxlib
import jax
from jax import lax
from jax import random as jrnd
from jax import numpy as jnp
from jax import tree_util as jtu
jax.config.update('jax_enable_x64', True)

# Others
from matplotlib import pyplot as plt

# This
from numerics import *
from seismic import *

In [2]:
@jtu.register_pytree_node_class
class GMMLT:
    """
    GMM Logic Tree (GMMLT)

    This class represents a logic tree for combining multiple Ground Motion Models (GMMs) 
    with associated weights for probabilistic seismic hazard analysis.

    Attributes
    ----------
    T : float
        The spectral period for which the ground motion is evaluated.
    gmms : list
        A list of GMM callable objects. Each GMM should accept the arguments 
        (Mw, T, site, fault, R) and return (mu_lnSA, sigma_lnSA).
    weights : jax.typing.ArrayLike
        An array of weights corresponding to each GMM in the logic tree.

    Methods
    -------
    calc_single(i: int, Mw: float, site: Site, fault: Fault, R: jax.Array)
        Computes the mean and standard deviation of lnSA for a single GMM specified by index i.

    tree_flatten()
        Flattens the GMMLT object for JAX pytree compatibility.

    tree_unflatten(aux, children)
        Reconstructs a GMMLT object from flattened components for JAX pytree compatibility.
    """
    def __init__(self, gmms:list, T:float, weights:jax.typing.ArrayLike):
        self.T = T
        self.gmms = gmms
        self.weights = weights

    def calc_single(self, i:int, Mw:float, site:Site, fault:Fault, R:jax.Array):
        """
        Computes the mean and standard deviation of the natural logarithm of spectral acceleration (lnSA)
        for a single Ground Motion Model (GMM) specified by its index.

        Parameters
        ----------
        i : int
            Index of the GMM in the logic tree to use for the calculation.
        Mw : float
            Moment magnitude of the earthquake.
        site : Site
            Site object containing site-specific parameters.
        fault : Fault
            Fault object containing fault-specific parameters.
        R : jax.Array
            Array of distances from the site to the fault.

        Returns
        -------
        mu_lnSA : float
            Mean of the natural logarithm of spectral acceleration.
        sigma_lnSA : float
            Standard deviation of the natural logarithm of spectral acceleration.
        """
        return lax.switch(i, self.gmms, Mw, self.T, site, fault, R)

    def tree_flatten(self):
        return (self.T, self.weights), self.gmms
    
    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(aux, *children)

@jtu.register_pytree_node_class
class HazCalculator:
    """
    Probabilistic Seismic Hazard Analysis (PSHA) Calculator

    This class provides methods for calculating seismic hazard using a logic tree of Ground Motion Models (GMMs).
    It supports incremental and marginal hazard calculations, ground motion evaluation, and epistemic uncertainty
    quantification for PSHA applications.

    Attributes
    ----------
    gmmlt : GMMLT
        The logic tree of Ground Motion Models (GMMs) with associated weights.
    dM : float, optional
        The magnitude bin size for discretizing the magnitude-frequency distribution (default is 0.01).

    Methods
    -------
    haz_calc_n_M_incr(scn: Scenario)
        Computes incremental rupture rates (number of events per bin) for all faults and magnitude bins.
    mhaz_calc_n_M_incr(M: float, scn: Scenario)
        Computes marginal incremental rupture rates for a specific magnitude M.
    haz_calc_all_lnSA(scn: Scenario)
        Calculates the mean and standard deviation of the natural logarithm of spectral acceleration (lnSA)
        for all GMMs, faults, and magnitude bins.
    mhaz_calc_all_lnSA(M: float, scn: Scenario)
        Calculates the mean and standard deviation of lnSA for all GMMs and faults at a specific magnitude M.
    haz_calc_haz_from_lnSA(im: jax.Array, all_mu_lnSA: jax.Array, all_std_lnSA: jax.Array, all_w: jax.Array, n_M_incr: jax.Array)
        Computes the total hazard curve (annual frequency of exceedance) for a given intensity measure (IM)
        using the computed lnSA statistics and incremental rates.
    haz_calc_epi_from_lnSA(im: jax.Array, all_mu_lnSA: jax.Array, all_std_lnSA: jax.Array, all_w: jax.Array, n_M_incr: jax.Array, ddof: int = 0)
        Computes epistemic uncertainty (weighted standard deviation) in the hazard curve due to logic tree weights.
    tree_flatten()
        Flattens the HazCalculator object for JAX pytree compatibility.
    tree_unflatten(aux, children)
        Reconstructs a HazCalculator object from flattened components for JAX pytree compatibility.
    """
    def __init__(self, gmmlt:GMMLT, dM:float = 0.01):
        """Initialize with GMMs and a bin size"""
        self.gmmlt = gmmlt
        self.dM = dM
        
    def haz_calc_n_M_incr(self, scn:Scenario):
        """
        Compute incremental rupture rates for all faults and magnitude bins.

        Parameters
        ----------
        scn : Scenario
            The scenario containing the fault tree and magnitude-frequency distribution.

        Returns
        -------
        M_min : jax.Array
            Minimum magnitude for each fault.
        M_max : jax.Array
            Maximum magnitude for each fault.
        bins_M : jax.Array
            Array of magnitude bin edges.
        n_M_incr : jax.Array
            Incremental rupture rates (number of events per bin) for each fault and magnitude bin.
        """
        # Magnitude bins.
        M_min, M_max = scn.fault_tree.mfd.M_min, scn.fault_tree.mfd.M_max
        bins_M = jnp.arange(M_min.min(), M_max.max() + self.dM, self.dM)
        # MFD rates
        n_M_exc = jax.vmap(scn.fault_tree.mfd.calc_lmdaM)(bins_M)
        # Take difference to convert from exceedance rates to discrete binned rates
        n_M_incr = n_M_exc[:-1] - n_M_exc[1:]
        return M_min, M_max, bins_M, n_M_incr
    
    def mhaz_calc_n_M_incr(self, M:float, scn:Scenario):
        """
        Compute marginal incremental rupture rates for a specific magnitude M.

        Parameters
        ----------
        M : float
            The magnitude at which to compute the marginal incremental rate.
        scn : Scenario
            The scenario containing the fault tree and magnitude-frequency distribution.

        Returns
        -------
        M_min : jax.Array
            Minimum magnitude for each fault.
        M_max : jax.Array
            Maximum magnitude for each fault.
        bins_M : jax.Array
            Magnitude bin edges for the given M.
        n_M_incr : jax.Array
            Marginal incremental rupture rates for the given magnitude.
        """
        M_min, M_max = scn.fault_tree.mfd.M_min, scn.fault_tree.mfd.M_max
        idx = jnp.floor((M - M_min) / self.dM)
        edge_idcs = idx + jnp.array([0, 1])
        bins_M = M_min + edge_idcs * self.dM
        n_M_exc = jax.vmap(scn.fault_tree.mfd.calc_lmdaM)(bins_M)
        n_M_incr = jnp.diff(n_M_exc)
        return M_min, M_max, bins_M, n_M_incr
    
    def haz_calc_all_lnSA(self, scn:Scenario):
        """
        Calculate mean and standard deviation of lnSA for all GMMs, faults, and magnitude bins.

        Parameters
        ----------
        scn : Scenario
            The scenario containing the site, fault tree, and magnitude-frequency distribution.

        Returns
        -------
        all_mu_lnSA : jax.Array
            Mean lnSA for each GMM, magnitude bin, fault, and site.
        all_std_lnSA : jax.Array
            Standard deviation of lnSA for each GMM, magnitude bin, fault, and site.
        n_M_incr : jax.Array
            Incremental rupture rates for each magnitude bin and fault.
        """
        M_min, M_max, bins_M, n_M_incr = self.haz_calc_n_M_incr(scn)
        # Roots for GMM evaluation.
        roots_M = (bins_M[:-1] + bins_M[1:]) / 2
        # Array of ones/zeros for each fault signifying array inside/outside range (shape (roots_M.shape, fault_num))
        weights_mask = (roots_M[:, None] > M_min[None, :]) & (roots_M[:, None] < M_max[None, :])
        # We can think of our incremental rates as already including quadrature weights, so 
        #   we'll just multiply them by the mask to be safe. 
        n_M_incr = n_M_incr * weights_mask

        # Ground motion means + stds for each fault at roots
        # Calculate R for full fault tree...
        R_tree = jax.vmap(calc_R, in_axes = (None, 0))(scn.site, scn.fault_tree)
        # Triple vmap. First, across faults (and corresponding distances),
        calc_faults = jax.vmap(self.gmmlt.calc_single, in_axes=(None, None, None, 0, 0))
        # Then across magnitudes,
        calc_M = jax.vmap(calc_faults, in_axes=(None, 0, None, None, None))
        # Then across GMMs. This order minimizes recompilation.
        calc_gmms = jax.vmap(calc_M, in_axes=(0, None, None, None, None))

        # Grab indices and vmap across
        gmm_idcs = jnp.arange(len(self.gmmlt.gmms))
        all_mu_lnSA, all_std_lnSA = calc_gmms(gmm_idcs, roots_M, scn.site, scn.fault_tree, R_tree)
        return all_mu_lnSA, all_std_lnSA, n_M_incr
    
    def mhaz_calc_all_lnSA(self, M:float, scn:Scenario):
        """
        Calculate mean and standard deviation of lnSA for all GMMs and faults at a specific magnitude M.

        Parameters
        ----------
        M : float
            The magnitude at which to compute lnSA.
        scn : Scenario
            The scenario containing the site and fault tree.

        Returns
        -------
        all_mu_lnSA : jax.Array
            Mean lnSA for each GMM and fault at magnitude M.
        all_std_lnSA : jax.Array
            Standard deviation of lnSA for each GMM and fault at magnitude M.
        n_M_incr : jax.Array
            Marginal incremental rupture rates for the given magnitude.
        """
        M_min, M_max, bins_M, n_M_incr = self.mhaz_calc_n_M_incr(scn)

        # Ground motion means + stds for each fault at roots
        # Calculate R for full fault tree...
        R_tree = jax.vmap(calc_R, in_axes = (None, 0))(scn.site, scn.fault_tree)
        # Triple vmap. First, across faults (and corresponding distances),
        calc_faults = jax.vmap(self.gmmlt.calc_single, in_axes=(None, None, None, 0, 0))
        # Then across GMMs. This order minimizes recompilation.
        calc_gmms = jax.vmap(calc_faults, in_axes=(0, None, None, None, None))

        # Grab indices and vmap across
        gmm_idcs = jnp.arange(len(self.gmmlt.gmms))
        all_mu_lnSA, all_std_lnSA = calc_gmms(gmm_idcs, M, scn.site, scn.fault_tree, R_tree)
        return all_mu_lnSA, all_std_lnSA, n_M_incr

    def haz_calc_haz_from_lnSA(self, im:jax.Array, 
                           all_mu_lnSA:jax.Array, all_std_lnSA:jax.Array, all_w:jax.Array, 
                           n_M_incr:jax.Array):
        """
        Compute the total hazard curve (annual frequency of exceedance) for a given intensity measure (IM).

        Parameters
        ----------
        im : jax.Array
            Array of intensity measure (IM) values (e.g., PGA = 0.2g).
        all_mu_lnSA : jax.Array
            Mean lnSA for each GMM, magnitude bin, fault, and site.
        all_std_lnSA : jax.Array
            Standard deviation of lnSA for each GMM, magnitude bin, fault, and site.
        all_w : jax.Array
            Weights for each GMM in the logic tree.
        n_M_incr : jax.Array
            Incremental rupture rates for each magnitude bin and fault.

        Returns
        -------
        jax.Array
            Hazard curve: annual frequency of exceedance for each im value.
        """
        # Get PoE at all points
        all_prob_x = 1 - jax.vmap(trunc_norm_cdf, in_axes = (0, None, None, None, None))(jnp.log(im), 
                                        -jnp.inf, 3 * all_std_lnSA, 
                                        all_mu_lnSA, all_std_lnSA)

        # Take mean
        mu_prob_x = jnp.einsum('i,hijk->hjk', all_w, all_prob_x)

        # Hazard integrand (magnitude probabilities * exceedance probabilities)
        haz_intgrnd = mu_prob_x * n_M_incr

        # Return the sum since our "quadrature weights" are basically
        #   included in our frequency bins
        return jnp.einsum('ijk->i',haz_intgrnd)
    
    def haz_calc_epi_from_lnSA(self, im:jax.Array, 
                           all_mu_lnSA:jax.Array, all_std_lnSA:jax.Array, all_w:jax.Array, 
                           n_M_incr:jax.Array,
                           ddof:int = 0):
        """
        Compute epistemic uncertainty (weighted standard deviation) in the hazard curve due to logic tree weights.

        Parameters
        ----------
        im : jax.Array
            Array of intensity measure (IM) values.
        all_mu_lnSA : jax.Array
            Mean lnSA for each GMM, magnitude bin, fault, and site.
        all_std_lnSA : jax.Array
            Standard deviation of lnSA for each GMM, magnitude bin, fault, and site.
        all_w : jax.Array
            Weights for each GMM in the logic tree.
        n_M_incr : jax.Array
            Incremental rupture rates for each magnitude bin and fault.
        ddof : int, optional
            Delta degrees of freedom for the weighted standard deviation (default is 0).

        Returns
        -------
        jax.Array
            Epistemic uncertainty (weighted standard deviation) in the hazard curve for each im value.
        """
        n = all_mu_lnSA.shape[0]
        # Get PoE at all points
        all_prob_x = 1 - jax.vmap(trunc_norm_cdf, in_axes = (0, None, None, None, None))(jnp.log(im), 
                                        -jnp.inf, 3 * all_std_lnSA, 
                                        all_mu_lnSA, all_std_lnSA)

        # Hazard integrand (magnitude probabilities * exceedance probabilities)
        all_haz_intgrnd = all_prob_x * n_M_incr[None, None]
        all_haz_intgrl = jnp.einsum('hijk->hi', all_haz_intgrnd)
        mu_haz_intgrl = (all_haz_intgrl @ all_w)

        # Weighted StD (default population)
        epi_num = ((all_haz_intgrl - mu_haz_intgrl[:, None])**2 @ all_w) ** (1 / 2)
        epi_denom = all_w.sum() - ddof * (all_w ** 2).sum() / all_w.sum()
        return epi_num / epi_denom
    
    def tree_flatten(self):
        return (self.gmmlt, self.dM), None
    
    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(*children)

Build scenario

In [ ]:
# Simple site
x_site, y_site = 0., 0.
vs30 = 760
z1p0, z2p5 = 1.3, 0.
site = Site(x_site, y_site, vs30, z1p0, z2p5, 0.)

# Fault 1 is large, faraway earthquakes; fault 2 is small, close
x_fault1, y_fault1 = 50., 50.
theta1, width1 = 225, 1.5
mfd1 = MFD(3, 1)
x_fault2, y_fault2 = -6., -8.
theta2, width2 = 30, 2.7
# Shared fault params
z_hyp = 1.5
z_tor = 1.
dip = 45
rake = 0.
# Fault 1
fault1 = Fault(x_fault1, x_fault2, z_hyp, z_tor, theta1, dip1, rake, width1, 0., mfd1)


Parameter Enumeration

In [ ]:
n_param_arr = 8

M_min, M_max = 3.5, 8.
M_arr = jnp.linspace(M_min, M_max, n_param_arr)
lnR_min, lnR_max = 0.1, 8.
lnR_arr = jnp.linspace(lnR_min, lnR_max, n_param_arr)
vs30min, vs30max = 150, 1080
vs30_arr = jnp.linspace(vs30min, vs30max, n_param_arr)
z_hyp_min, z_hyp_max = 0.1, 18
z_hyp_arr = jnp.linspace(z_hyp_min, z_hyp_max, n_param_arr)

